In [1]:
import numpy as np
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

import torchvision
import torchvision.transforms as transforms

In [2]:
transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])


In [3]:
from torchvision.datasets import ImageFolder

train_data = ImageFolder(root='dataset/train', transform=transform)
test_data = ImageFolder(root='dataset/test', transform=transform)


In [4]:
train_loader = torch.utils.data.DataLoader(train_data, batch_size=32, shuffle=True, num_workers=2)
test_loader = torch.utils.data.DataLoader(test_data, batch_size=32, shuffle=False, num_workers=2)

In [5]:
image, label = train_data[0]
image.size()

torch.Size([3, 192, 192])

In [6]:
class_names = train_data.classes
class_names

['bus', 'car', 'motorcycle', 'train', 'truck']

In [7]:
class NeuralNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 16, 5)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(16, 32, 5)
        # 192 -> 188 (conv1) -> 94 (pool) -> 90 (conv2) -> 45 (pool)
        self.fc1 = nn.Linear(32 * 45 * 45, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = torch.flatten(x, 1)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x


In [8]:
net = NeuralNet()
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.SGD(net.parameters(), lr=0.001, momentum=0.9)

In [9]:
for epoch in range(30):
    print(f'Training  epoch {epoch}...')
    
    running_loss = 0.0
    
    for i, data in enumerate(train_loader):
        inputs, labels = data
        
        optimizer.zero_grad()
        
        outputs = net(inputs)
        
        loss = loss_fn(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        
    print(f'Loss: {running_loss / len(train_loader):.4f}')

Training  epoch 0...
Loss: 1.6567
Training  epoch 1...
Loss: 1.6567
Training  epoch 1...
Loss: 1.3934
Training  epoch 2...
Loss: 1.3934
Training  epoch 2...
Loss: 1.3049
Training  epoch 3...
Loss: 1.2313
Training  epoch 4...
Loss: 1.1613
Training  epoch 5...
Loss: 1.1613
Training  epoch 5...
Loss: 1.1249
Training  epoch 6...
Loss: 1.1249
Training  epoch 6...
Loss: 1.0719
Training  epoch 7...
Loss: 1.0719
Training  epoch 7...
Loss: 1.0250
Training  epoch 8...
Loss: 1.0250
Training  epoch 8...
Loss: 0.9975
Training  epoch 9...
Loss: 0.9780
Training  epoch 10...
Loss: 0.9338
Training  epoch 11...
Loss: 0.8906
Training  epoch 12...
Loss: 0.8710
Training  epoch 13...
Loss: 0.8376
Training  epoch 14...
Loss: 0.8016
Training  epoch 15...
Loss: 0.7695
Training  epoch 16...
Loss: 0.7494
Training  epoch 17...
Loss: 0.7256
Training  epoch 18...
Loss: 0.6776
Training  epoch 19...
Loss: 0.6590
Training  epoch 20...
Loss: 0.6310
Training  epoch 21...
Loss: 0.6022
Training  epoch 22...
Loss: 0.5651
T

In [10]:
torch.save(net.state_dict(), 'model.pth')

In [11]:
net = NeuralNet()
net.load_state_dict(torch.load('model.pth'))

<All keys matched successfully>

In [12]:
correct = 0
total = 0

net.eval()

with torch.no_grad():
    for data in test_loader:
        images, labels = data
        outputs = net(images)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
print(f'Accuracy: {100 * correct / total:.2f}%')

Accuracy: 68.79%


In [14]:
new_tranform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

def load_image(image_path):
    
    image = Image.open(image_path)
    image = new_tranform(image).unsqueeze(0)
    return image

image_paths = ['dataset/validation/car/1.png', 'dataset/validation/truck/2.png', 'dataset/validation/bus/3.png']

images = [load_image(image_path) for image_path in image_paths]

net.eval()
with torch.no_grad():
    for i, image in enumerate(images):
        outputs = net(image)
        _, predicted = torch.max(outputs, 1)
        print(f'Image {i+1} is predicted as {class_names[predicted.item()]}')


Image 1 is predicted as truck
Image 2 is predicted as car
Image 3 is predicted as bus
